### 1. Input

In [ ]:
# ! pip install SpeechRecognition pydub
# ! pip install langdetect
# ! pip install openai-whisper
# ! pip install torch
# ! pip install pyaudio
# ! pip install ollama
import speech_recognition as sr
import whisper as w
import torch
from langdetect import detect
import re
import ollama

#### 1.1. S2T

In [2]:
device = "mps" if torch.backends.mps.is_available() else "cpu"
# Modelo base
model = w.load_model("base", device=device)

def input_speech(audio = None):
    # Opción 1: Se le pasa un archivo de audio .wav
    if audio != None:
        # Utilizamos whisper porque procesa el archivo directamente, detecta el idioma y lo transcribe (con speech_recognition había que especificar el idioma previamente)
        result = model.transcribe(audio)
        # Se devuelve la transcripción del audio y el idioma
        return result["text"].strip(), result["language"]
    
    # Opción 2: Audio en directo
    else:
        r = sr.Recognizer()
        with sr.Microphone() as source:
            # Para reconocer el nivel de ruido en el audio y eliminar esa parte
            r.adjust_for_ambient_noise(source, duration=1)
            r.pause_threshold = 2.0
            # Se escucha el audio
            audio_data = r.listen(source, phrase_time_limit=20)
            
            # Guardamos el audio temporalmente ya que whisper lo necesita
            with open("audio_temporal.wav", "wb") as a:
                a.write(audio_data.get_wav_data())
                
            result = model.transcribe("audio_temporal.wav")
            # Devolvemos el texto y el idioma
            return result["text"].strip(), result["language"]

#### 1.2. T2T

In [3]:
def input_text():
    # El usuario escribe sobre qué quiere información
    texto = input("¿Sobre qué quieres información?")

    # Detectamos el idioma
    try:
        idioma = detect(texto)
    except:
        # Idioma por defecto
        idioma = "es"
    
    print(f"Idioma del texto: {idioma}")
    # Devolvemos el texto y el idioma
    return texto, idioma

#### 1.3. Cleaning text

In [4]:
def cleaning_text(texto, idioma):
    # strip para eliminar espacios en blanco
    # capitalize para poner la primera letra en mayúscula
    texto_limpio = texto.strip().capitalize()

    # Muletillas a eliminar (español e inglés)
    muletillas = {
        "es": [r"\btipo+\b", r"\beh+\b", r"\besto+\b", r"\bpues+\b", r"\ben\s+plan\b"],
        "en": [r"\bum+\b", r"\beh+\b", r"\blike\b", r"\byou know\b"]
    }
    
    # Eliminamos las muletillas
    if idioma in muletillas:
        for i in muletillas[idioma]:
            texto_limpio = re.sub(i, "", texto_limpio, flags = re.IGNORECASE)

    # Eliminamos los n espacios que se han creado al eliminar las muletillas y lo sustitumos por un espacio
    texto_limpio = re.sub(r"\s+", " ", texto_limpio).strip()

    return texto_limpio

### 2. Generar respuesta (T2T)

In [16]:
def respuesta_texto(texto_limpio, idioma):
    print("\n--- Opciones: ---")
    print("1. Resumen breve")
    print("2. Historia detallada")
    print("3. Datos curiosos")
    
    estilo_op = input("\nSelecciona el tipo de información deaseada (1, 2 o 3): ")

    if estilo_op == "1":
        estilo = "Hazme un resumen breve."
    elif estilo_op == "2":
        estilo = "Hazme una explicación detallada y técnica."
    else:
        estilo = "Cuéntame curiosidades."

    prompt = f"""
    Eres un guía turístico experto de clase mundial. 
    Tu objetivo es hablar sobre: {texto_limpio}.
    Instrucción específica: {estilo}
    IMPORTANTE: Debes responder OBLIGATORIAMENTE en el idioma: {idioma}.
    """

    respuesta_ia = ollama.chat(model = "llama3", messages=[{'role': 'user', 'content': prompt}])
    
    return respuesta_ia['message']['content']

In [ ]:
def menu_principal():
    print("\n--- Guía turística---")
    print("1. Escribir")
    print("2. Hablar/Audio")
    print("3. Hacer foto")
    print("4. Salir")
    
    opcion = input("\nSelecciona una opción (1, 2, 3 o 4): ")

    texto_sucio, idioma = None, None
    if opcion == "1":
        texto_sucio, idioma = input_text()
    
    elif opcion == "2":
        print("\n¿Quieres (a) hablar o (b) pasar un archivo .wav?")
        op = input("Selecciona una opción (a o b): ").lower()
        if op == "a":
            print("Escuchando...")
            texto_sucio, idioma = input_speech()
        else:
            ruta = input("Introduce la ruta .wav: ")
            texto_sucio, idioma = input_speech(audio=ruta)
            
    elif opcion == "3":
        return
        
    elif opcion == "4":
        print("...")
        return
    else:
        print("Opción no válida.")
        return

    if texto_sucio and idioma:
        texto_limpio = cleaning_text(texto_sucio, idioma)
        print(f"\n Procesando texto: {texto_limpio}")

        respuesta_final = respuesta_texto(texto_limpio, idioma)
        
        print("\n Respuesta:")
        print(respuesta_final)        
        return respuesta_final